# Approximation Algorithms for Steiner Trees in Weighted Graphs

Nikola Labus — Naučno izračunavanje 2025/26

In [6]:
import networkx as nx
import time
from itertools import combinations
from pathlib import Path

## STP Parser

In [7]:
def parse_stp(filepath):
    G = nx.Graph()
    terminals = []
    name = ""
    section = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('33D32945'):
                continue

            if line.startswith('SECTION'):
                section = line.split()[1]
                continue
            if line == 'END' or line == 'EOF':
                section = None
                continue

            if section == 'Comment':
                if line.startswith('Name'):
                    name = line.split('"')[1]

            elif section == 'Graph':
                if line.startswith('Nodes'):
                    n = int(line.split()[1])
                    G.add_nodes_from(range(1, n + 1))
                elif line.startswith('E '):
                    parts = line.split()
                    u, v, w = int(parts[1]), int(parts[2]), int(parts[3])
                    G.add_edge(u, v, weight=w)

            elif section == 'Terminals':
                if line.startswith('T '):
                    terminals.append(int(line.split()[1]))

    return G, terminals, name

## Brute Force (Egzaktan algoritam)

Za svaki podskup neterminalnih čvorova proveravamo da li se terminali mogu povezati
kroz indukovani podgraf. Pamtimo minimalno razapinjuće stablo sa najmanjom težinom.

In [8]:
def brute_force_steiner(G, terminals):
    terminal_set = set(terminals)
    non_terminals = [v for v in G.nodes() if v not in terminal_set]
    best_weight = float('inf')
    best_tree = None
    subsets_checked = 0

    for k in range(len(non_terminals) + 1):
        for subset in combinations(non_terminals, k):
            subsets_checked += 1
            nodes = terminal_set | set(subset)
            subgraph = G.subgraph(nodes)

            if nx.is_connected(subgraph):
                mst = nx.minimum_spanning_tree(subgraph)
                weight = mst.size(weight='weight')
                if weight < best_weight:
                    best_weight = weight
                    best_tree = mst

    return best_tree, best_weight, subsets_checked

2## MST Heuristika (Kou, Markowsky, Berman)

Konstruišemo kompletni graf od terminala gde su težine jednake najkraćim putevima u originalnom grafu.
Na tom grafu nalazimo MST, a zatim zamenjujemo grane originalnim putevima i uklanjamo cikluse i nepotrebne listove.

In [12]:
def mst_heuristic(G, terminals):
    complete = nx.Graph()
    paths = {}
    for i, t1 in enumerate(terminals):
        for t2 in terminals[i + 1:]:
            length, path = nx.single_source_dijkstra(G, t1, t2)
            complete.add_edge(t1, t2, weight=length)
            paths[(t1, t2)] = path


    mst = nx.minimum_spanning_tree(complete)

    steiner = nx.Graph()
    for u, v in mst.edges():
        key = (u, v) if (u, v) in paths else (v, u)
        path = paths[key]
        for j in range(len(path) - 1):
            w = G[path[j]][path[j + 1]]['weight']
            steiner.add_edge(path[j], path[j + 1], weight=w)

    steiner = nx.minimum_spanning_tree(steiner)

    changed = True
    while changed:
        changed = False
        leaves = [v for v in steiner.nodes() if steiner.degree(v) == 1 and v not in terminals]
        for leaf in leaves:
            steiner.remove_node(leaf)
            changed = True
    weight = steiner.size(weight='weight')
    return steiner, weight